In [1]:
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer
from formatting import processing_function, split_dataset

SYNTAX_DATASET_PATH = "dataset//base//syntax_examples.jsonl"
DOMAIN_DATASET_PATH = "dataset//base//domain_examples.jsonl"
SAVE_FOLDER_PATH = "dataset" 

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048

SYNTAX_DATASET_PATH = Path(SYNTAX_DATASET_PATH).resolve()
DOMAIN_DATASET_PATH = Path(DOMAIN_DATASET_PATH).resolve()
SAVE_FOLDER_PATH = Path(SAVE_FOLDER_PATH).resolve() 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# To avoid recompiling entire dataset, skip to step 8 and just load the data, trim and split it

In [2]:
# Step 1: Create Syntax examples
df_syntax = pd.read_json(SYNTAX_DATASET_PATH, lines=True)
print(f"Created syntax mistakes dataset with {len(df_syntax)} entries.")

Created syntax mistakes dataset with 5497 entries.


In [3]:
# Step 2: Create domain examples
df_domain = pd.read_json(DOMAIN_DATASET_PATH, lines=True)
print(f"Created domain mistakes dataset with {len(df_domain)} entries.")

Created domain mistakes dataset with 1402 entries.


In [4]:
# Step 3: Create correct examples
df_correct = df_domain.drop_duplicates(subset="source_id", keep="first").copy()

df_correct["mutation_type"] = "none"
df_correct["mutation_category"] = "none"
df_correct["error_message"] = "none"
df_correct["bad_code"] = df_correct["good_code"]

# Using 30:70 correct vs incorrect distribution, and repeating correct examples accordingly
freq = round((0.3/0.7) * (len(df_syntax) + len(df_domain))/len(df_correct))
df_correct = pd.concat([df_correct] * freq, ignore_index=True)
print(f"Created correct entries dataset with a repetition frequency of {freq}, and {len(df_correct)} entries.")

Created correct entries dataset with a repetition frequency of 3, and 3057 entries.


In [ ]:
# Step 4: Concenate
df = pd.concat([df_syntax, df_domain, df_correct], ignore_index=True)

before = len(df)

# Drop exact duplicates (all columns)
df = df.drop_duplicates()

dropped = before - len(df)

print(
    f"Datasets merged. Final dataset contains {len(df)} entries "
    f"({dropped} duplicates dropped)."
)

Datasets merged. Final dataset contains 9956 entries.


In [ ]:
# Step 5: Add prompts etc, and ids
ds = Dataset.from_pandas(df).remove_columns("__index_level_0__")
ds = ds.map(
    processing_function,
    batched=False,
    fn_kwargs={"tokenizer": tokenizer},
)

def add_ids(example, idx):
    return {"id": idx + 1}

ds = ds.map(add_ids, with_indices=True)
cols = ["id"] + [c for c in ds.column_names if c != "id"]
ds = ds.select_columns(cols)
ds = ds.shuffle(seed=42)

path = Path(SAVE_FOLDER_PATH) / "compiled_dataset.jsonl"
ds.to_json(path, lines=True)

print(f"Dataset saved to {path}")

Map:   0%|          | 0/9956 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Dataset saved to C:\Data\Research\Doctoral\INCOSE\code\incose\dataset\compiled_dataset.jsonl


In [7]:
# Step 8: Drop longer examples
path = Path(SAVE_FOLDER_PATH) / "compiled_dataset.jsonl"
ds = Dataset.from_pandas(pd.read_json(path, lines=True))
len_before = len(ds)
ds = ds.filter(lambda x: x["length"] <= MAX_SEQ_LENGTH)
len_after = len(ds)

print(f"Dropped {len_before-len_after} entries due to being longer than {MAX_SEQ_LENGTH} tokens.")
print(f"Final dataset size: {len_after} entries.")

Filter:   0%|          | 0/9956 [00:00<?, ? examples/s]

Dropped 525 entries due to being longer than 2048 tokens.
Final dataset size: 9431 entries.


In [8]:
# Step 9: Split datasets and save them
train_ds, val_ds, test_ds = split_dataset(ds)
    
train_ds.to_json(SAVE_FOLDER_PATH / "split" / "train_dataset.jsonl", lines=True)
val_ds.to_json(SAVE_FOLDER_PATH / "split" / "eval_dataset.jsonl", lines=True)
test_ds.to_json(SAVE_FOLDER_PATH / "split" / "test_dataset.jsonl", lines=True)

print(f"Split datasets saved to {SAVE_FOLDER_PATH}")

Split sizes: train=6649, val=1402, test=1380


Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Split datasets saved to C:\Data\Research\Doctoral\INCOSE\code\incose\dataset


In [10]:
train_ds[0]

{'id': 2563,
 'source_id': 'src_0733',
 'mutation_category': 'syntax',
 'mutation_type': 'swap_keywords',
 'bad_code': 'package Vehicle_Remix_0574_4fdc {\n    part def MotorInput;\n    port def HandPort;\n    part def MotorInput_Def { port p : MotorInput; }\n    part def MotorInput_Def { port p : MotorInput; }\n    part def HandPort_Distractor_Def { port p : HandPort; }\n    part def SubSystem_Context {\n        part comp_a_f436 : MotorInput_Def;\n        part comp_b_7253 : MotorInput_Def;\n        part comp_distractor_a5d9 : HandPort_Distractor_Def;\n        connect comp_a_f436.p to comp_b_7253.p;\n    }\n}',
 'error_message': 'ERROR:A port must be typed by port definitions. (6049.sysml line : 8 column : 31) \nERROR:A port must be typed by port definitions. (6049.sysml line : 9 column : 31)',
 'good_code': 'package Vehicle_Remix_0574_4fdc {\n    port def MotorInput;\n    port def HandPort;\n    part def MotorInput_Def { port p : MotorInput; }\n    part def MotorInput_Def { port p : Mo